# 十二类主任务：T4 分步骤运行

每格调用独立的 `.py` 文件。数据已准备，无需重采集。开发14106张、留出5904张；分组CV评分覆盖6类，留出覆盖8类，不能宣称完整十二类独立评测。

将 `uav12_code.zip`、`uav12_images.zip`、`transfer_receipt.json` 放入 Drive 的 `MyDrive/ml_hw_uav12/`。选择GPU运行时，按顺序执行。压缩包只用于传输，内部保留模块目录。


## 1 挂载 Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
TRANSFER = Path('/content/drive/MyDrive/ml_hw_uav12')
ROOT = Path('/content/ml_hw_uav12')


## 2 核对并展开代码


In [ ]:
import hashlib, json, os, shutil, sys, zipfile
receipt = json.loads((TRANSFER/'transfer_receipt.json').read_text(encoding='utf-8-sig'))
code_zip = TRANSFER/'uav12_code.zip'
def file_hash(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024*1024), b''):
            digest.update(chunk)
    return digest.hexdigest()
if file_hash(code_zip) != receipt['archives']['uav12_code.zip']['sha256']:
    raise ValueError('代码包上传不完整或版本不一致')
ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(code_zip) as archive:
    for item in archive.infolist():
        target = (ROOT/item.filename).resolve()
        if not target.is_relative_to(ROOT.resolve()) or '\\' in item.filename or ':' in item.filename or ((item.external_attr >> 16)&0o170000)==0o120000:
            raise ValueError('不安全的压缩包路径')
    archive.extractall(ROOT)
os.environ['ML_PROJECT_ROOT'] = str(ROOT)
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
sys.path.insert(0, str(ROOT/'src'))
print('代码已展开：', ROOT)


## 3 安装同一套依赖

在模型步骤之前执行一次；不要在开发和留出之间升级包。保留 Colab 已有的兼容 CUDA torch/torchvision。


In [ ]:
import subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(ROOT/'requirements-uav12-qwen.txt')], check=True)


## 4 展开图片、恢复结果、核对实际GPU

T4默认FP16、单图训练、梯度累积8步。Qwen每250个优化步直接保存可恢复检查点到Drive，其余产物每一步结束后同步。断线后重新执行1至4，再运行尚未完成的步骤。


In [ ]:
from uav_project.uav12_colab import unpack_images, ColabUAV12
unpack_images(TRANSFER, ROOT, receipt)
session = ColabUAV12(ROOT, TRANSFER, receipt['transfer_id'], receipt['experiment_path'])
session.restore()
session.record_environment()


## 5 开发集手工特征

CPU步骤，只处理开发集。


In [ ]:
session.run('06d_extract_uav12_features.py', '--representation', 'handcrafted')


## 6 开发集 SigLIP2 表征

冻结编码器、固定模型commit；GPU只提取特征。显存不足可将16改8重跑本格。


In [ ]:
session.run('06d_extract_uav12_features.py', '--representation', 'siglip2', '--batch-size', '16')


## 7 手工特征四分类器和网格搜索

开发集两折分组CV，6类评分、12类训练；标准化和筛选在训练折内拟合。


In [ ]:
session.run('07d_train_uav12_baselines.py', '--representation', 'handcrafted', '--jobs', '2')


## 8 SigLIP2表征四分类器和网格搜索

与上一格使用相同分组折和评分范围。CPU运行是正常现象。


In [ ]:
session.run('07d_train_uav12_baselines.py', '--representation', 'siglip2', '--jobs', '2')


## 9 Qwen普通监督微调

QLoRA只适配语言模块；视觉骨干冻结；固定一轮开发集监督学习，不使用留出集挑检查点。首次下载基础权重。


In [ ]:
session.run('07q_train_qwen3vl.py', '--variant', 'standard')


## 10 Qwen类别加权监督微调

与普通组从同一基础模型开始，使用同一样本和步数，仅改变归一化的平方根逆频率损失权重。不是强化学习。


In [ ]:
session.run('07q_train_qwen3vl.py', '--variant', 'sqrt_class_balanced')


## 11 全局冻结

全部已声明模型完成才允许冻结。冻结后不能更改配置、提示词、训练参数或权重。


In [ ]:
session.run('09d_freeze_uav12.py')


## 12 冻结后的留出手工特征

未冻结时程序拒绝读取。


In [ ]:
session.run('06d_extract_uav12_features.py', '--representation', 'handcrafted', '--partition', 'holdout')


## 13 冻结后的留出SigLIP2表征

模型commit、代码和数据身份与开发阶段对应。


In [ ]:
session.run('06d_extract_uav12_features.py', '--representation', 'siglip2', '--partition', 'holdout', '--batch-size', '16')


## 14 一次性冻结评估

报告全部传统模型、原始Qwen和两个LoRA版本；非法输出按错误计入。每250张保存Qwen中间预测以支持续跑。


In [ ]:
session.run('10d_evaluate_uav12.py')


## 15 结果表与图

从实际输出生成CSV、混淆矩阵、特征重要性及Markdown，不生成PDF。


In [ ]:
session.run('11d_export_uav12_report.py')


## 16 下载结果

把结果ZIP交回来，用于继续写论文。模型适配器、特征矩阵和训练检查点继续保留在Drive，不必打包下载。


In [ ]:
from google.colab import files
result_zip = session.export_results()
files.download(str(result_zip))


本Notebook只生成，未在本地执行训练、模型推理或软件测试。首次GPU运行仍需确认依赖与显存；出现报错停在对应格，发完整错误，修复后按检查点续跑。当前是单种子、固定训练预算的课程研究实验，不能提前声称性能提升或论文创新。
